# SVM vs. BERTimbau vs. SetFit Comparação de desempenho (Nível 1 e Nível 2)

Mini-paper comparando o baseline **TF-IDF + LinearSVC**, o **BERTimbau fine-tuned** e o **SetFit** na classificação de tickets ITIL do AutoITIL:

- **Nível 1** categoria ITIL (Incidente, Requisição, Problema, Mudança): SVM vs. BERT
- **Nível 2** área técnica responsável (Aplicações, Servidor, Rede, Segurança, BD): SVM vs. BERT vs. SetFit

**Este notebook é 100% autossuficiente.** Se os 5 arquivos de resultado (`svm_nivel1_cv_results.json`, `bert_nivel1_cv_results.json`, `svm_nivel2_cv_results.json`, `bert_nivel2_cv_results.json`, `setfit_nivel2_cv_results.json`) já existirem no ambiente (local ou upload prévio no Colab), ele só carrega sem retreinar nada. Se não existirem, ele treina os 5 modelos com CV 5-fold ali mesmo e gera os JSONs. **Rode com GPU habilitada no Colab** (`Ambiente de execução → Alterar tipo de ambiente de execução → GPU`) sem GPU o CV do BERT e principalmente do SetFit (contrastive fine-tuning gera ~15k pares por treino) ficam lentos.

## Metodologia

Os modelos são avaliados sob as **mesmas condições** em cada nível, para que a comparação seja justa:

- Mesmo dataset (185 tickets, IDs idênticos entre `dataset.csv` e `dataset_nivel_2.csv`), mesma seed (42).
- **Stratified 5-Fold CV com predições out-of-fold (OOF)** cada ticket é predito exatamente uma vez, no fold em que caiu como teste. Evita o problema do split único de 20%, que deixa classes minoritárias (ex.: BD, 13 exemplos) com 2-3 exemplos de validação muito pouco para uma métrica estável.
- Todos tratam o desbalanceamento de classes: SVM via `class_weight="balanced"` no `LinearSVC`, BERT via `CrossEntropyLoss` ponderada por classe com o mesmo cálculo de peso (`WeightedLossTrainer`), SetFit via contrastive fine-tuning (menos sensível a desbalanceamento por natureza).
- Cada modelo usa o **texto no formato para o qual foi desenhado**: SVM usa `descricao_limpa` (TF-IDF se beneficia de texto normalizado) BERT e SetFit usam `descricao` (texto cru ambos se beneficiam de texto natural).
- Métrica principal: **F1 macro** (dá peso igual a todas as classes, inclusive as minoritárias mais informativo que accuracy num dataset desbalanceado).

> **Nota sobre reprodutibilidade:** a seed é fixada, mas treino de rede neural em GPU não é 100% determinístico (cuDNN). Se você rodar este notebook e os números saírem um pouco diferentes dos citados no texto, é esperado os JSONs gerados são a fonte da verdade, não o texto.


In [ ]:
!pip install -q transformers==4.57.6 datasets==5.0.0 scikit-learn==1.9.0 accelerate==1.14.0 setfit==1.1.3 sentence-transformers==5.6.0

## 1. Carregar os dois datasets


In [ ]:
from pathlib import Path

import pandas as pd


def load_csv(nome_arquivo):
    candidatos = [
        Path(f"data/{nome_arquivo}"),
        Path(f"../data/{nome_arquivo}"),
    ]
    local = next((c for c in candidatos if c.exists()), None)
    if local is not None:
        print(f"Lido localmente: {local}")
        return pd.read_csv(local)

    from google.colab import files

    print(f"Selecione o arquivo: {nome_arquivo}")
    uploaded = files.upload()
    return pd.read_csv(list(uploaded.keys())[0])


df_n1 = load_csv("dataset.csv")
df_n2 = load_csv("dataset_nivel_2.csv")

# Compatibilidade: aceita tanto 'area_responsavel' quanto o antigo 'nivel2'
if "area_responsavel" not in df_n2.columns and "nivel2" in df_n2.columns:
    df_n2 = df_n2.rename(columns={"nivel2": "area_responsavel"})

print("\nNível 1: ", len(df_n1), "tickets")
print(df_n1["categoria"].value_counts())
print("\nNível 2: ", len(df_n2), "tickets")
print(df_n2["area_responsavel"].value_counts())

## 2. Setup compartilhado (seed, tokenizer, helpers de treino)


In [ ]:
import gc
import json
import os

import numpy as np
import torch
from datasets import Dataset
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

os.environ["WANDB_DISABLED"] = "true"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 192
N_FOLDS = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(
        batch["text"], truncation=True, padding=True, max_length=MAX_LENGTH
    )

In [ ]:
def build_svm_pipeline():
    """TF-IDF + LinearSVC calibrado mesmo baseline nos dois níveis."""
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
                ),
            ),
            (
                "clf",
                CalibratedClassifierCV(
                    LinearSVC(max_iter=2000, class_weight="balanced"), cv=3
                ),
            ),
        ]
    )


def run_svm_cv(df, target_col, text_col, tag):
    df = df.dropna(subset=[target_col, text_col]).reset_index(drop=True)
    classes = sorted(df[target_col].unique())
    X = np.array(df[text_col].fillna("").tolist(), dtype=object)
    y = np.array(df[target_col].tolist(), dtype=object)

    oof_pred = np.empty(len(df), dtype=object)
    fold_f1 = []
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        pipe = build_svm_pipeline()
        pipe.fit(X[tr_idx].tolist(), y[tr_idx].tolist())
        preds = pipe.predict(X[te_idx].tolist())
        oof_pred[te_idx] = preds
        f1 = f1_score(y[te_idx], preds, average="macro", zero_division=0)
        fold_f1.append(f1)
        print(f"[{tag}] Fold {fold}: F1 macro = {f1:.3f}")

    cv_results = {
        "model": "tfidf+linearsvc",
        "n_folds": N_FOLDS,
        "text_col": text_col,
        "f1_macro_per_fold": [round(f, 4) for f in fold_f1],
        "f1_macro_mean": round(float(np.mean(fold_f1)), 4),
        "f1_macro_std": round(float(np.std(fold_f1)), 4),
        "oof_classification_report": classification_report(
            y,
            oof_pred,
            labels=classes,
            target_names=classes,
            output_dict=True,
            zero_division=0,
        ),
    }
    print(
        f"[{tag}] F1 macro CV: {cv_results['f1_macro_mean']:.4f} +/- {cv_results['f1_macro_std']:.4f}"
    )
    return cv_results

In [ ]:
class WeightedLossTrainer(Trainer):
    """Trainer com CrossEntropyLoss ponderada por classe (mitiga desbalanceamento)."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = (
            self._class_weights.to(logits.device)
            if self._class_weights is not None
            else None
        )
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def run_bert_cv(df, target_col, text_col, batch_size, epochs_cv, tag):
    df = df.dropna(subset=[target_col, text_col]).reset_index(drop=True)
    le = LabelEncoder()
    df = df.copy()
    df["label"] = le.fit_transform(df[target_col])
    num_labels = len(le.classes_)
    id2label = {i: c for i, c in enumerate(le.classes_)}
    label2id = {c: i for i, c in id2label.items()}

    X = df[text_col].values
    y = df["label"].values
    oof_pred = np.full(len(df), -1, dtype=int)
    fold_f1 = []
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        tr = df.iloc[tr_idx]
        te = df.iloc[te_idx]
        tr_ds = Dataset.from_pandas(
            tr[[text_col, "label"]].rename(columns={text_col: "text"}),
            preserve_index=False,
        ).map(tokenize, batched=True)
        te_ds = Dataset.from_pandas(
            te[[text_col, "label"]].rename(columns={text_col: "text"}),
            preserve_index=False,
        ).map(tokenize, batched=True)

        w = torch.tensor(
            compute_class_weight(
                class_weight="balanced",
                classes=np.arange(num_labels),
                y=tr["label"].values,
            ),
            dtype=torch.float,
        )
        fold_model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
        )
        fold_args = TrainingArguments(
            output_dir=f"cv_fold{fold}_{tag}",
            num_train_epochs=epochs_cv,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=32,
            learning_rate=2e-5,
            weight_decay=0.01,
            eval_strategy="no",
            save_strategy="no",
            logging_strategy="no",
            seed=RANDOM_SEED,
            report_to="none",
        )
        fold_trainer = WeightedLossTrainer(
            model=fold_model,
            args=fold_args,
            train_dataset=tr_ds,
            tokenizer=tokenizer,
            class_weights=w,
        )
        fold_trainer.train()
        preds = np.argmax(fold_trainer.predict(te_ds).predictions, axis=-1)
        oof_pred[te_idx] = preds
        f1 = f1_score(te["label"].values, preds, average="macro", zero_division=0)
        fold_f1.append(f1)
        print(f"[{tag}] Fold {fold}: F1 macro = {f1:.3f}")

        del fold_model, fold_trainer
        gc.collect()
        torch.cuda.empty_cache()

    cv_results = {
        "n_folds": N_FOLDS,
        "epochs_cv": epochs_cv,
        "f1_macro_per_fold": [round(f, 4) for f in fold_f1],
        "f1_macro_mean": round(float(np.mean(fold_f1)), 4),
        "f1_macro_std": round(float(np.std(fold_f1)), 4),
        "oof_classification_report": classification_report(
            y,
            oof_pred,
            target_names=list(le.classes_),
            output_dict=True,
            zero_division=0,
        ),
    }
    print(
        f"[{tag}] F1 macro CV: {cv_results['f1_macro_mean']:.4f} +/- {cv_results['f1_macro_std']:.4f}"
    )
    return cv_results

In [ ]:
SETFIT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def run_setfit_cv(df, target_col, text_col, tag):
    """Contrastive fine-tuning (SetFit) Trainer/TrainingArguments próprios do
    pacote setfit, por isso importados aqui (não confundir com os do transformers)."""
    from setfit import SetFitModel
    from setfit import Trainer as SetFitTrainer
    from setfit import TrainingArguments as SetFitTrainingArguments

    df = df.dropna(subset=[target_col, text_col]).reset_index(drop=True)
    classes = sorted(df[target_col].unique())
    X = np.array(df[text_col].tolist(), dtype=object)
    y = np.array(df[target_col].tolist(), dtype=object)

    oof_pred = np.empty(len(df), dtype=object)
    fold_f1 = []
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        tr_ds = Dataset.from_dict(
            {"text": X[tr_idx].tolist(), "label": y[tr_idx].tolist()}
        )

        fold_model = SetFitModel.from_pretrained(SETFIT_MODEL_NAME)
        fold_args = SetFitTrainingArguments(
            batch_size=16, num_epochs=1, seed=RANDOM_SEED, report_to="none"
        )
        fold_trainer = SetFitTrainer(
            model=fold_model, args=fold_args, train_dataset=tr_ds
        )
        fold_trainer.train()

        preds = fold_model.predict(X[te_idx].tolist())
        oof_pred[te_idx] = preds
        f1 = f1_score(y[te_idx], preds, average="macro", zero_division=0)
        fold_f1.append(f1)
        print(f"[{tag}] Fold {fold}: F1 macro = {f1:.3f}")

        del fold_model, fold_trainer
        gc.collect()
        torch.cuda.empty_cache()

    cv_results = {
        "model": f"setfit+{SETFIT_MODEL_NAME.split('/')[-1]}",
        "n_folds": N_FOLDS,
        "text_col": text_col,
        "f1_macro_per_fold": [round(f, 4) for f in fold_f1],
        "f1_macro_mean": round(float(np.mean(fold_f1)), 4),
        "f1_macro_std": round(float(np.std(fold_f1)), 4),
        "oof_classification_report": classification_report(
            y,
            oof_pred,
            labels=classes,
            target_names=classes,
            output_dict=True,
            zero_division=0,
        ),
    }
    print(
        f"[{tag}] F1 macro CV: {cv_results['f1_macro_mean']:.4f} +/- {cv_results['f1_macro_std']:.4f}"
    )
    return cv_results

## 3. Gerar (ou carregar) os 5 resultados

Se o JSON de um modelo já existe no ambiente, ele é só carregado, sem retreinar. Caso contrário, o CV é executado agora e o JSON é salvo.


In [ ]:
RESULT_SPECS = {
    ("SVM", "Nível 1"): {
        "filename": "svm_nivel1_cv_results.json",
        "fn": lambda: run_svm_cv(df_n1, "categoria", "descricao_limpa", tag="svm_n1"),
    },
    ("BERT", "Nível 1"): {
        "filename": "bert_nivel1_cv_results.json",
        "fn": lambda: run_bert_cv(
            df_n1, "categoria", "descricao", batch_size=16, epochs_cv=4, tag="bert_n1"
        ),
    },
    ("SVM", "Nível 2"): {
        "filename": "svm_nivel2_cv_results.json",
        "fn": lambda: run_svm_cv(
            df_n2, "area_responsavel", "descricao_limpa", tag="svm_n2"
        ),
    },
    ("BERT", "Nível 2"): {
        "filename": "bert_nivel2_cv_results.json",
        "fn": lambda: run_bert_cv(
            df_n2,
            "area_responsavel",
            "descricao",
            batch_size=8,
            epochs_cv=6,
            tag="bert_n2",
        ),
    },
    ("SetFit", "Nível 2"): {
        "filename": "setfit_nivel2_cv_results.json",
        "fn": lambda: run_setfit_cv(
            df_n2, "area_responsavel", "descricao", tag="setfit_n2"
        ),
    },
}

results = {}
for key, spec in RESULT_SPECS.items():
    path = Path(spec["filename"])
    if path.exists():
        print(f"{key} -> {path} já existe, carregando do disco (sem retreinar).")
        with open(path) as f:
            results[key] = json.load(f)
    else:
        print(f"{key} -> {path} não encontrado, treinando agora...")
        results[key] = spec["fn"]()
        with open(path, "w") as f:
            json.dump(results[key], f, indent=2, ensure_ascii=False)
        print(f"{key} -> salvo em {path}")

list(results.keys())

## 4. Tabela comparativa F1 macro (CV 5-fold, OOF)


In [ ]:
rows = []
for (model, nivel), r in results.items():
    rows.append(
        {
            "Nível": nivel,
            "Modelo": model,
            "F1 macro (média)": r["f1_macro_mean"],
            "F1 macro (desvio)": r["f1_macro_std"],
            "Accuracy (OOF)": round(r["oof_classification_report"]["accuracy"], 4),
        }
    )

summary = pd.DataFrame(rows).sort_values(["Nível", "Modelo"]).reset_index(drop=True)
summary

In [ ]:
for nivel in ["Nível 1", "Nível 2"]:
    svm_f1 = results[("SVM", nivel)]["f1_macro_mean"]
    bert_f1 = results[("BERT", nivel)]["f1_macro_mean"]
    delta = svm_f1 - bert_f1
    vencedor = "SVM" if delta > 0 else "BERT" if delta < 0 else "empate"
    print(
        f"{nivel}: SVM={svm_f1:.4f}  BERT={bert_f1:.4f}  delta(SVM-BERT)={delta:+.4f}  vencedor={vencedor}"
    )

if ("SetFit", "Nível 2") in results:
    svm_f1 = results[("SVM", "Nível 2")]["f1_macro_mean"]
    setfit_f1 = results[("SetFit", "Nível 2")]["f1_macro_mean"]
    delta = svm_f1 - setfit_f1
    vencedor = "SVM" if delta > 0 else "SetFit" if delta < 0 else "empate"
    print(
        f"Nível 2: SVM={svm_f1:.4f}  SetFit={setfit_f1:.4f}  delta(SVM-SetFit)={delta:+.4f}  vencedor={vencedor}"
    )

## 5. Gráfico comparativo


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 4.5))

niveis = ["Nível 1", "Nível 2"]
modelos = ["SVM", "BERT"]
COLOR = {"SVM": "#4C72B0", "BERT": "#DD8452"}
x = np.arange(len(niveis))
width = 0.32

for i, modelo in enumerate(modelos):
    means = [results[(modelo, n)]["f1_macro_mean"] for n in niveis]
    stds = [results[(modelo, n)]["f1_macro_std"] for n in niveis]
    offset = (i - 0.5) * width
    ax.bar(
        x + offset,
        means,
        width,
        yerr=stds,
        capsize=4,
        label=modelo,
        color=COLOR[modelo],
    )

ax.set_xticks(x)
ax.set_xticklabels(niveis)
ax.set_ylabel("F1 macro (CV 5-fold, média ± desvio)")
ax.set_ylim(0, 1)
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(title="Modelo")
ax.set_title("SVM vs. BERT F1 macro por nível")
fig.tight_layout()
plt.show()

## 5b. Extra SetFit no Nível 2

O SetFit ataca especificamente o problema de poucos exemplos por classe (contrastive fine-tuning de sentence-transformer). Só existe para o Nível 2 (`setfit_nivel2_training.ipynb`), então entra num gráfico separado comparar categoricamente 3 modelos num nível só, em vez de forçar a mesma grade dos 2 níveis anteriores.


In [ ]:
if ("SetFit", "Nível 2") in results:
    modelos_n2 = ["SVM", "BERT", "SetFit"]
    COLOR_N2 = {"SVM": "#4C72B0", "BERT": "#DD8452", "SetFit": "#55A868"}
    means = [results[(m, "Nível 2")]["f1_macro_mean"] for m in modelos_n2]
    stds = [results[(m, "Nível 2")]["f1_macro_std"] for m in modelos_n2]

    fig, ax = plt.subplots(figsize=(6, 4.5))
    x = np.arange(len(modelos_n2))
    ax.bar(x, means, yerr=stds, capsize=4, color=[COLOR_N2[m] for m in modelos_n2])
    ax.set_xticks(x)
    ax.set_xticklabels(modelos_n2)
    ax.set_ylabel("F1 macro (CV 5-fold, média ± desvio)")
    ax.set_ylim(0, 1)
    ax.set_title("Nível 2 SVM vs. BERT vs. SetFit")
    fig.tight_layout()
    plt.show()
else:
    print("SetFit ausente pulei o gráfico de 3 modelos.")

## 6. Discussão

### Nível 1 categoria ITIL

Resultado real (execução confirmada em Colab GPU): **SVM 0.8684 ± 0.0423** vs. **BERT 0.7592 ± 0.0748**. SVM na frente por +0.1092, e com desvio bem menor entre folds. Fatores que explicam isso num dataset deste tamanho (185 tickets):

- **Sinal lexical forte por classe.** As quatro categorias ITIL (Incidente, Requisição, Problema, Mudança) têm marcadores textuais bem definidos verbos de "solicito"/"preciso" (Requisição), "aprovad[a]"/RFC (Mudança), "recorrente"/"causa raiz" (Problema) algo que um TF-IDF de 1-2 gramas captura diretamente, sem precisar de contexto semântico profundo.
- **185 exemplos é pouco para o BERT aprender representações contextuais específicas do domínio**, mas costuma bastar para o SVM, que tem muito menos parâmetros e generaliza melhor nesse regime (ver `docs/pesquisa_finetuning_bert_dataset_pequeno.md` cenário descrito por Sun et al., 2019: baseline clássico competitivo com transformer em poucos dados).
- Na matriz de confusão do BERT (ver `bert_nivel1_training.ipynb`), o maior erro é Incidente confundido com Problema (17 casos) o SVM erra bem menos nesse par.

### Nível 2 área técnica responsável

Resultado real: **SVM 0.6513 ± 0.0999**, **BERT 0.5224 ± 0.1209**, **SetFit 0.616 ± 0.0942** (quando presente). SetFit fica entre os dois supera o BERT fine-tuned completo por +0.0936, mas ainda não alcança o SVM (delta -0.0353). Isso é coerente com a proposta do SetFit: mitiga o overfitting do BERT via contrastive fine-tuning, mas não chega a superar um TF-IDF bem regularizado com este volume de dados.

O BERT também é o modelo mais instável dos três nesse nível desvio de 0.1209 entre folds, com o fold 5 caindo pra 0.325 (contra 0.685 no fold 1). Isso reforça que 5 classes desbalanceadas com poucos exemplos (BD = 13) é um regime particularmente difícil pra um modelo com tantos parâmetros livres quanto o BERT.

Por que o Nível 2 é mais difícil que o Nível 1 para os três modelos:

- **5 classes em vez de 4**, com desbalanceamento mais severo (86 vs. 13 no extremo, contra 68 vs. 34 no Nível 1) a classe minoritária (BD) fica com poucos exemplos por fold de teste.
- **Sobreposição semântica real entre classes**: um ticket de VPN pode ser Rede ou Segurança, um ticket de lentidão pode ser Servidor ou Aplicações a ambiguidade está no problema, não só no modelo. Isso tende a penalizar mais modelos com muitos parâmetros livres (BERT, e em menor grau o SetFit) tentando aprender fronteiras finas com poucos exemplos por classe.
- Nenhuma das abordagens usa hierarquia entre os níveis (ex.: usar a categoria Nível 1 predita como feature para o Nível 2) possível extensão futura.

## Conclusão

1. **SVM supera BERT nos dois níveis** (+0.1092 no Nível 1, +0.1289 no Nível 2) e supera o SetFit no Nível 2 (+0.0353) com menor variância entre folds em todos os casos. É o modelo mais forte e mais estável neste dataset de 185 tickets.
2. **SetFit > BERT fine-tuned completo** no Nível 2 (0.616 vs. 0.5224) confirma que técnicas desenhadas para poucos exemplos por classe (contrastive fine-tuning) ajudam mais que fine-tuning tradicional de um BERT inteiro neste regime de dados, mesmo sem superar o baseline clássico.
3. Isso não é incomum em classificação de texto **com datasets pequenos (<500 exemplos)**: fine-tuning de transformers tende a overfitar mais rápido que um classificador linear regularizado sobre TF-IDF.
4. **Implicação prática para o AutoITIL:** o `CascadeClassifier` deve continuar usando SVM como modelo de produção nos dois níveis não há, neste dataset, ganho de qualidade de nenhuma abordagem baseada em transformer que justifique o custo de infraestrutura de GPU.
5. **Próximo passo natural:** aplicar ao BERT as técnicas de regularização de `docs/pesquisa_finetuning_bert_dataset_pequeno.md` (layer freezing, layer-wise LR decay, label smoothing) mesmo assim, dado a folga do SVM, o objetivo seria mais empatar do que superar o baseline.


## 7. Baixar os JSONs gerados (Colab)

Traga os 5 arquivos de volta para `notebooks/` no repo, para versionar os resultados.


In [ ]:
try:
    from google.colab import files

    for spec in RESULT_SPECS.values():
        files.download(spec["filename"])
except ImportError:
    print("Não é Colab os JSONs já estão salvos localmente em notebooks/.")